# QUESTION 6 — QUELS 3 000 SPECTATEURS CIBLER PAR SOIRÉE ?

## Problématique

Si on ne peut cibler que 3 000 spectateurs par soirée
avec une offre, lesquels ? Une offre envoyée à quelqu'un
qui ne consomme jamais est une offre perdue.

## Objectifs

- Estimer la propension des spectateurs à consommer au bar.
- Classer les spectateurs selon leur score de consommation.
- Comparer plusieurs stratégies de ciblage.
- Limiter le nombre d'offres envoyées aux non-consommateurs.
- Proposer une stratégie adaptée à la contrainte des
  3 000 spectateurs par soirée.

## Lien avec la question 5

La question 5 a montré les limites du KNN pour la
prédiction individuelle de la consommation au bar.

L'analyse complémentaire a cependant révélé des différences
de consommation selon les zones de la salle.

La question 6 exploite ces résultats pour construire
et évaluer différentes stratégies de ciblage commercial.

## 1. Importation et chargement des données

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Chargement des données nettoyées
billets = pd.read_csv('../data/processed/billets_clean.csv')

evenements = pd.read_csv('../data/processed/evenements_clean.csv',parse_dates=['date'])

# Vérification des dimensions
print("Dimensions des billets :", billets.shape)
print("Dimensions des événements :", evenements.shape)

Dimensions des billets : (700, 15)
Dimensions des événements : (96, 9)


## 2. Préparation des données

Les billets dont la consommation au bar est inconnue
sont exclus. La variable cible et les variables explicatives
retenues dans la question 5 sont réutilisées.

Le nombre de billets vendus est également récupéré pour
simuler la contrainte de ciblage des 3 000 spectateurs,
mais ne sera pas utilisé comme variable prédictive.

In [3]:
# Exclusion des consommations inconnues
donnees_q6 = billets.dropna(subset=['conso_bar_eur']).copy()

# Création de la variable cible
donnees_q6['conso_bar'] = (donnees_q6['conso_bar_eur'] > 0).astype(int)

# Jointure avec les données événementielles
donnees_q6 = donnees_q6.merge(
    evenements[[
        'evenement_id', 'genre', 'jour_semaine',
        'capacite', 'notoriete_artiste', 'billets_vendus'
    ]],
    on='evenement_id',
    how='left',
    validate='many_to_one'
)

# Sélection des variables explicatives
variables_numeriques = [
    'prix_paye_eur', 'achat_jours_avant',
    'age', 'notoriete_artiste'
]

variables_categorielles = [
    'zone', 'genre', 'jour_semaine', 'capacite'
]

variables_booleennes = ['deja_venu', 'invitation']

variables_explicatives = (
    variables_numeriques
    + variables_categorielles
    + variables_booleennes
)

# Vérification des résultats
print("Billets exploitables :", len(donnees_q6))
print("Consommateurs :", donnees_q6['conso_bar'].sum())
print("Non-consommateurs :", (donnees_q6['conso_bar'] == 0).sum())
print("Nombre de variables explicatives :", len(variables_explicatives))
print("Billets sans fréquentation connue :", donnees_q6['billets_vendus'].isna().sum())

Billets exploitables : 686
Consommateurs : 614
Non-consommateurs : 72
Nombre de variables explicatives : 10
Billets sans fréquentation connue : 22


## 3. Analyse de la contrainte des 3 000 spectateurs

L'échantillon ne contient pas suffisamment de billets
pour sélectionner directement 3 000 spectateurs par soirée.

Un quota proportionnel au nombre de billets vendus est
donc calculé pour chaque événement, puis appliqué au nombre
de billets exploitables de l'échantillon.

Les événements dont le nombre de ventes est inconnu
sont exclus de cette évaluation.

Cette approche constitue une simulation rétrospective
et non un ciblage réel de 3 000 spectateurs.

In [4]:
# Conserver les billets dont le nombre de ventes est connu et positif
donnees_ciblage = donnees_q6[
    donnees_q6['billets_vendus'].notna()
    & (donnees_q6['billets_vendus'] > 0)
].copy()

# Regroupement des billets par événement
quotas_soirees = donnees_ciblage.groupby('evenement_id').agg(
    billets_vendus=('billets_vendus', 'first'),
    taille_echantillon=('billet_id', 'count')
)

# Proportion de spectateurs à cibler
quotas_soirees['proportion_ciblage'] = (
    3000 / quotas_soirees['billets_vendus']
).clip(upper=1)

# Calcul du quota dans l'échantillon (arrondi au plus proche)
quotas_soirees['quota_ciblage'] = np.floor(
    quotas_soirees['taille_echantillon']
    * quotas_soirees['proportion_ciblage'] + 0.5
).astype(int)

# Synthèse des résultats
print("Billets retenus :", len(donnees_ciblage))
print("Événements retenus :", len(quotas_soirees))
print("Nombre total de billets à cibler :", quotas_soirees['quota_ciblage'].sum())
print("Événements avec un quota nul :", (quotas_soirees['quota_ciblage'] == 0).sum())

display(quotas_soirees.head(10).round(3))

Billets retenus : 664
Événements retenus : 91
Nombre total de billets à cibler : 375
Événements avec un quota nul : 0


,billets_vendus,taille_echantillon,proportion_ciblage,quota_ciblage
evenement_id,,,,
EV300,4861.0,9,0.617,6
EV301,6285.0,5,0.477,2
EV302,3639.0,4,0.824,3
EV304,5009.0,10,0.599,6
EV305,5674.0,9,0.529,5
EV306,5068.0,5,0.592,3
EV307,5047.0,9,0.594,5
EV308,6833.0,7,0.439,3
EV309,4916.0,5,0.610,3


**Interprétation :**

L'évaluation porte sur 664 billets répartis sur 91 événements.
Les stratégies devront sélectionner 375 billets, soit 56,48 %
de l'échantillon retenu, en respectant les quotas par soirée.

## 4. Définition du protocole d'évaluation

Une validation croisée stratifiée et groupée par événement
en cinq plis est utilisée pour éviter le chevauchement
des soirées entre entraînement et validation.

Les mêmes plis et les mêmes quotas sont utilisés pour
comparer équitablement toutes les stratégies de ciblage.

In [5]:
from sklearn.model_selection import StratifiedGroupKFold

# Préparation des données pour les modèles
donnees_ciblage = donnees_ciblage.reset_index(drop=True)

X = donnees_ciblage[variables_explicatives]
y = donnees_ciblage['conso_bar']
groupes = donnees_ciblage['evenement_id']

# Validation croisée groupée
cv_groupes = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

plis_cv = list(cv_groupes.split(X, y, groups=groupes))

# Vérification des plis
verification_plis = []

for i, (train_idx, val_idx) in enumerate(plis_cv, start=1):

    groupes_train = set(groupes.iloc[train_idx])
    groupes_val = set(groupes.iloc[val_idx])

    verification_plis.append({
        'Pli': i,
        'Billets entraînement': len(train_idx),
        'Billets validation': len(val_idx),
        'Événements entraînement': len(groupes_train),
        'Événements validation': len(groupes_val),
        'Non-consommateurs validation': (y.iloc[val_idx] == 0).sum(),
        'Événements communs': len(groupes_train & groupes_val)
    })

display(pd.DataFrame(verification_plis))

,Pli,Billets entraînement,Billets validation,Événements entraînement,Événements validation,Non-consommateurs validation,Événements communs
0,1,533,131,74,17,14,0
1,2,532,132,73,18,14,0
2,3,530,134,73,18,15,0
3,4,530,134,72,19,14,0
4,5,531,133,72,19,14,0


**Interprétation :**

Les cinq plis présentent des effectifs équilibrés et
aucun événement commun entre entraînement et validation.

La classe minoritaire est représentée dans chaque pli.
Ce protocole permet de comparer les stratégies de ciblage
sur des événements non utilisés pour leur entraînement.